In [3]:
from bs4 import BeautifulSoup
import os
import time
import requests
from urllib.parse import urlparse

In [10]:
# --- CONFIGURATION ---
TEAM_URL = "https://www.basketmaniacs.com/team/unleash-the-clowns/"


SEASONS = {
    1: "2023-24",
    2: "2024-25",
    3: "2025-26"
}

# 1. Extract Team Name from URL (e.g., 'unleash-the-clowns')
parsed_url = urlparse(TEAM_URL)
team_slug = parsed_url.path.strip('/').split('/')[-1]
print(f" Target Team: {team_slug}")

# 2. Create the specific folder
base_dir = os.path.dirname(os.getcwd())
print(f"Base Directory: {base_dir}")
raw_data_dir = os.path.join(base_dir, 'data', 'raw', team_slug)

os.makedirs(raw_data_dir, exist_ok=True)

print(f"Folder ready: {raw_data_dir}")

 Target Team: unleash-the-clowns
Base Directory: /Users/nikos/basketball-stats
Folder ready: /Users/nikos/basketball-stats/data/raw/unleash-the-clowns


In [ ]:
players_to_download = []

# Loop through every season defined in Block 1
for season_id, season_name in SEASONS.items():
    print(f"\n Scanning Season: {season_name} (ID: {season_id})...")
    
    # Add ?season=X to get the specific roster for that year
    season_team_url = f"{TEAM_URL}?season={season_id}"
    print(f" URL: {season_team_url}")
    

    try:
        response = requests.get(season_team_url)
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find all player cards
        roster_items = soup.find_all('div', class_='team-roster__item')
        print(f"Scanning roster... Found {len(roster_items)} cards.\n")

        for item in roster_items:
            # 1. Find the Link (<a> tag)
            link_tag = item.find('a', href=True)
            if not link_tag:
                continue # Skip if no link found (e.g., empty card)
                
            player_url = link_tag['href']
            if not player_url.startswith('http'):
                    player_url = "https://www.basketmaniacs.com" + player_url
            # IMPORTANT: Append season ID to player URL too!
            # This ensures we get stats for 2023, not the current year
            final_player_url = f"{player_url}?season={season_id}"
            
            # 2. Find the Name (<h2> tag)
            name_tag = item.find('h2', class_='team-roster__member-name')
            
            if name_tag:
                # Clean the name: Remove newlines, extra spaces
                raw_name = name_tag.get_text(separator=' ', strip=True)
                clean_name = " ".join(raw_name.split())
            else:
                # Fallback if name is missing
                clean_name = "Unknown_Player_" + player_url.split('/')[-1]

            # Add to our list
            players_to_download.append({
                'name': clean_name,
                'url': final_player_url,
                'season': season_name
            })
        
    except Exception as e:
        print(f"Error scanning {season_name}: {e}")

print(f"\n Ready to download {len(players_to_download)} players.")


 Scanning Season: 2023-24 (ID: 1)...
 URL: https://www.basketmaniacs.com/team/unleash-the-clowns/?season=1
Scanning roster... Found 19 cards.


 Scanning Season: 2024-25 (ID: 2)...
 URL: https://www.basketmaniacs.com/team/unleash-the-clowns/?season=2
Scanning roster... Found 19 cards.


 Scanning Season: 2025-26 (ID: 3)...
 URL: https://www.basketmaniacs.com/team/unleash-the-clowns/?season=3
Scanning roster... Found 19 cards.


 Ready to download 57 players.


In [ ]:
import time
import re
import random


# 1. The "Mask" (Pretend to be a Browser)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def download_player(player_data):
    name = player_data['name']
    url = player_data['url']
    season = player_data['season'] 

    # 1. Create Season Subfolder
    season_dir = os.path.join(raw_data_dir, season)
    os.makedirs(season_dir, exist_ok=True)
    
    # Create clean filename (e.g., "Nikos_Vassakis.html")
    # This regex keeps Greek letters, English letters, and numbers
    safe_name = re.sub(r'[^\w\u0370-\u03FF_-]', '', name.replace(" ", "_"))
    filename = f"{safe_name}.html"
    save_path = os.path.join(season_dir, filename)
    
    # Skip if we already have it
    if os.path.exists(save_path):
        print(f" [{season}] Skipping {name} (Already downloaded)")
        return

    print(f"[{season}] Downloading: {name}...", end=" ")
    
    try:
        # Pass the HEADERS here
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status() # Check for errors (404, 500)
        
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write(response.text)
        print("✅ Done!")
        
        # Pause for a random time between 3-6 seconds to be polite to the server
        sleep_time = random.uniform(3, 6)
        print(f"   zzz... Sleeping for {sleep_time:.2f}s...")
        time.sleep(sleep_time)
        
    except Exception as e:
        print(f" Error: {e}")

# --- EXECUTE ---
print(f"Starting download for {len(players_to_download)} players...")
print("-" * 40)

for player in players_to_download:
    download_player(player)

print("-" * 40)
print(f"Success! All files are in: {raw_data_dir}")

Starting batch download for 19 players...
----------------------------------------
⏩ Skipping Παναγιώτης Ζέρβας (Already downloaded)
⬇️  Downloading: Απόλλων Γραμματικόπουλος... ✅ Done!
   zzz... Sleeping for 5.70s...
⬇️  Downloading: Pavlos Glynos... ✅ Done!
   zzz... Sleeping for 4.95s...
⬇️  Downloading: Στεφανος Μεγαλος... ✅ Done!
   zzz... Sleeping for 5.79s...
⬇️  Downloading: Ηλίας Χαλκίτης... ✅ Done!
   zzz... Sleeping for 5.10s...
⬇️  Downloading: Θεόδωρος Κάγκας... ✅ Done!
   zzz... Sleeping for 3.68s...
⬇️  Downloading: Νίκος Λάμπρου... ✅ Done!
   zzz... Sleeping for 5.39s...
⬇️  Downloading: Φραγκισκος Βαμβακαρης... ✅ Done!
   zzz... Sleeping for 3.14s...
⬇️  Downloading: Νικόλαος Βασσάκης... ✅ Done!
   zzz... Sleeping for 3.99s...
⬇️  Downloading: Γιαννης Μαγειροπουλος... ✅ Done!
   zzz... Sleeping for 5.94s...
⬇️  Downloading: Νότης Φακλης... ✅ Done!
   zzz... Sleeping for 3.78s...
⬇️  Downloading: Σταύρος Παπαδόπουλος... ✅ Done!
   zzz... Sleeping for 3.37s...
⬇️  Downlo